# 模型调用的两种方式

基于 langchain==1.3.0，本文先聚焦 LangChain 中最常见的两种模型访问方式：
- invoke：阻塞式访问，等待完整结果返回
- stream：流式访问，边生成边返回

本文只围绕这两个接口，以及它们在 Agent 中的使用形式展开。

另外，该系列已发布过的 [02 | 基于 OpenAI Python SDK的模型接入方式]，其讲的是“怎么把模型服务接进来”，重点是 api_key、base_url、model 这些接入参数。而本文讲的是“进入 LangChain 之后，怎么调用模型”，重点是 invoke 和 stream 这两种调用方式。

## 一、先看结论

invoke 和 stream 的区别不在于“模型不同”，而在于“结果返回方式不同”。

- invoke：一次性返回完整结果
- stream：逐段返回结果，可用于实时输出

如果只是脚本调用、批处理、结果不需要实时展示，通常直接使用 invoke。

如果要做聊天界面、终端实时打印、长回答逐步展示，通常使用 stream。


In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 加载 .env 中的模型访问配置
load_dotenv()

# 这里使用本地Ollama模型举例
llm = init_chat_model(
    "ollama:gemma4:26b"
)

## 二、invoke：阻塞式访问

invoke 会一直等待，直到模型生成完整结果后再返回。

返回值通常是一个 AIMessage 对象；如果是聊天模型，常用 response.content 取文本内容。


In [ ]:
# invoke：等待完整回答返回
response = llm.invoke("请用一句话介绍 LangChain。")

# 返回的是标准消息对象，而不是纯字符串
print(type(response))
print(response.content)

invoke 的特点：
- 写法最简单
- 适合绝大多数后端逻辑
- 调试方便
- 但用户必须等到完整结果生成完毕后才能看到输出


## 三、stream：流式访问

stream 会返回一个可迭代对象，模型生成内容时可以逐段读取。

对于聊天模型，流式返回的通常是 AIMessageChunk。常见做法是把每个 chunk 的 content 逐步打印出来。


In [ ]:
# stream：边生成边输出
for chunk in llm.stream("请用三句话说明LangChain是什么？"):
    # 只有当前 chunk 带文本内容时才打印
    if chunk.content:
        print(chunk.content, end="",flush=True)
print()


stream 的特点：
- 适合聊天 UI、终端实时输出
- 用户可以更早看到模型开始响应
- 更适合长文本回答场景
- 处理逻辑比 invoke 稍复杂


## 四、同一个模型，invoke 和 stream 只是访问方式不同

下面这个对比最关键：模型实例本身没有变，变化的只是调用方法。


In [ ]:
# 同一个 llm，分别用 invoke 和 stream 对比返回方式
question = "请列出 LangChain 的 3 个典型用途。三句话说明。"

print("=== invoke ===")
result = llm.invoke(question)
print(result.content)

print("\n=== stream ===")
for chunk in llm.stream(question):
    # 流式场景下，内容会按块逐步到达
    if chunk.content:
        print(chunk.content, end="", flush=True)

print()


## 五、在 Agent 中使用模型的形式

在 LangChain Agent 中，模型通常不是手动反复调用 llm.invoke(...)，而是作为 create_agent(...) 的 model 参数传入。

本文先演示 create_agent 中两种最常见的传法：
- 直接传模型字符串，如 "deepseek:deepseek-v4-flash"
- 传一个已经初始化好的 chat model 实例


In [ ]:
from langchain.agents import create_agent

# 用一个最小工具函数演示 Agent 如何决定调用外部工具
def get_weather(city: str) -> str:
    """返回指定城市的天气信息。"""
    return f"{city} 今天天气晴，气温 25 度。"


In [ ]:
# 方式一：直接传模型字符串
agent = create_agent(
    model="ollama:gemma4:26b",
    tools=[get_weather],
    system_prompt="你是一个简洁、准确的天气助手。",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "北京今天天气怎么样？"}]}
)

# 这里打印的是整个 Agent state，而不只是最终回答
print(result)


上面这里的 agent.invoke(...)，调用的已经不是“单次模型问答”，而是“整个 Agent 运行过程”。

也就是说：
- Agent 内部仍然会调用模型
- 但对外暴露的是 Agent 自己的 invoke / stream
- 调用粒度已经从“模型”升级为“Agent”


In [ ]:
# 方式二：传入已经初始化好的模型实例
agent_llm = init_chat_model(
    "ollama:gemma4:26b",
    temperature=0,
)

agent2 = create_agent(
    model=agent_llm,
    tools=[get_weather],
    system_prompt="你是一个简洁、准确的天气助手。",
)

result2 = agent2.invoke(
    {"messages": [{"role": "user", "content": "上海今天天气怎么样？"}]}
)

# 和上面一样，返回值仍然是 Agent 的完整状态
print(result2)


## 六、Agent 中的 stream

Agent 也支持 stream。但这里的流式输出不只是“模型 token”，还可能是“Agent 的步骤更新”。

在官方文档里，常见写法是：
- stream_mode="updates"：看 Agent 每一步的进展
- stream_mode="messages"：看模型返回的增量消息块（通常可理解为文本流，也可能包含 tool call 等信息）


In [ ]:
# 只打印 Agent 执行过程中的关键信息
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "杭州今天天气怎么样？"}]},
    stream_mode="updates",
):
    # model 节点里能看到模型是否发起了 tool call，或给出了最终回复
    if "model" in chunk:
        model_messages = chunk["model"].get("messages", [])
        if model_messages:
            message = model_messages[-1]
            if getattr(message, "tool_calls", None):
                for tool_call in message.tool_calls:
                    print(f"[模型决定调用工具] {tool_call['name']} args={tool_call['args']}")
            elif getattr(message, "content", None):
                print(f"[模型回复] {message.content}")

    # tools 节点里能看到工具执行结果
    if "tools" in chunk:
        tool_messages = chunk["tools"].get("messages", [])
        for message in tool_messages:
            print(f"[工具结果] {message.content}")


In [ ]:
# 只观察 Agent 中模型的 token 流
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "杭州今天天气怎么样？请详细回答。"}]},
    stream_mode="messages",
):
    # 这里只演示最常见的文本输出；复杂场景下也可能出现 tool call 相关块
    if token.content:
        print(token.content, end="", flush=True)

print()


如果只是记最核心的区别，可以这样理解：

- llm.invoke(...)：调用一次模型，等完整结果
- llm.stream(...)：调用一次模型，边生成边返回
- agent.invoke(...)：运行一次 Agent，等完整结果
- agent.stream(...)：运行一次 Agent，并按 stream_mode 持续返回过程信息


## 七、总结

在 langchain==1.3.0 中，本文先把最常见的模型访问方式压缩成两类：

1. invoke
   - 阻塞式
   - 一次性拿完整结果
   - 适合大多数业务逻辑

2. stream
   - 流式
   - 边生成边返回
   - 适合交互式界面和长输出场景

在 Agent 中，模型通常作为 create_agent 的 model 参数传入；真正对外调用时，使用的是 Agent 自己的 invoke 和 stream。其中 stream 的具体返回形态，还取决于 stream_mode。
